# 🤖 Tarkari AI Assistant — Fine-Tuning Notebook
## Llama 3.2 3B Instruct + LoRA + Unsloth

**Supports:** Alpaca format, ShareGPT format, OpenAI Chat format  
**Works on:** T4 (Colab Free) and A100 (Colab Pro+)  
**Runtime:** Go to `Runtime > Change runtime type > GPU (T4 or A100)` before running

---
### 🚀 Quick Start
1. Set Runtime to GPU (T4 minimum)
2. Add your HuggingFace token to Colab Secrets (key icon in left panel) as `HF_TOKEN`
3. Run all cells top to bottom
4. Edit **Cell 4 (Configuration)** to change model, dataset, and training settings

In [ ]:
# ════════════════════════════════════════════════════
# CELL 1: Mount Google Drive
# ════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_PATH = "/content/drive/MyDrive/TarkariAI_Training"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
print(f"✅ Drive mounted. Outputs → {DRIVE_SAVE_PATH}")

In [ ]:
# ════════════════════════════════════════════════════
# CELL 2: Check GPU
# ════════════════════════════════════════════════════
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ════════════════════════════════════════════════════
# CELL 3: Install Dependencies (~3 min on first run)
# Uses Unsloth auto-installer to pick the correct xformers/torch combo
# ════════════════════════════════════════════════════
import subprocess, sys, os

# BUG FIX: The old method pinned xformers==0.0.29.post3 which breaks when
# Colab updates PyTorch. The auto-installer detects the right versions.
if "COLAB_" in "".join(os.environ.keys()):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "--no-deps", "bitsandbytes", "accelerate", "peft", "trl",
        "triton", "cut_cross_entropy", "unsloth_zoo"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "sentencepiece", "protobuf", "datasets>=3.4.1", "huggingface_hub", "hf_transfer"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "--no-deps", "unsloth"])
    # Install xformers compatible with current torch (auto-detected)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xformers"])
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unsloth"])

print("✅ All dependencies installed!")


In [ ]:
# ════════════════════════════════════════════════════
# CELL 4: ⚙️  CONFIGURATION — Edit these settings!
# ════════════════════════════════════════════════════

# ── Model ──────────────────────────────────────────
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
# Alternatives:
# "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"     # Microsoft Phi-3
# "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"         # Alibaba Qwen 2.5
# "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"       # Smaller/faster

# ── Dataset ────────────────────────────────────────
DATASET_SOURCE = "huggingface"    # "huggingface" | "local" | "upload"
DATASET_NAME   = "mlabonne/FineTome-100k"
DATASET_FORMAT = "sharegpt"       # "sharegpt" | "alpaca" | "openai"
MAX_SAMPLES    = 3000             # None = use full dataset
LOCAL_DATASET_PATH = "/content/drive/MyDrive/my_dataset.jsonl"

# ── LoRA Config ────────────────────────────────────
LORA_R       = 16    # 8=fast, 16=balanced, 32=quality
LORA_ALPHA   = 32    # Usually 2x rank
LORA_DROPOUT = 0.05

# ── Training ───────────────────────────────────────
MAX_SEQ_LENGTH = 2048   # Reduce to 1024 if OOM
BATCH_SIZE     = 2      # 2 for T4, 4-8 for A100
GRAD_ACCUM     = 4      # Effective batch = BATCH_SIZE * GRAD_ACCUM
NUM_EPOCHS     = 1
MAX_STEPS      = 200    # -1 for full training
LEARNING_RATE  = 2e-4
WARMUP_RATIO   = 0.05
WEIGHT_DECAY   = 0.01
SEED           = 3407
OUTPUT_DIR     = "/content/outputs"

# ── HuggingFace Hub ─────────────────────────────────
HF_USERNAME  = "your-hf-username"        # Your HF username
HF_REPO_NAME = "my-llama-3.2-finetuned" # Repo name on HF Hub
PUSH_TO_HUB  = False   # Set True after configuring HF token

print("✅ Configuration ready!")
print(f"   Model:   {MODEL_NAME}")
print(f"   Dataset: {DATASET_NAME} [{DATASET_FORMAT}]")
print(f"   Samples: {MAX_SAMPLES}")
print(f"   LoRA rank: {LORA_R} | Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Max steps: {MAX_STEPS}")

In [ ]:
# ════════════════════════════════════════════════════
# CELL 5: HuggingFace Login
# Add HF_TOKEN to Colab Secrets (left panel key icon)
# ════════════════════════════════════════════════════
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ Logged in to HuggingFace via Colab secret!")
except Exception as e:
    print(f"⚠️  No HF_TOKEN secret: {e}")
    print("   Add via the key icon in the left sidebar.")
    # Fallback: login(token='hf_YOUR_TOKEN_HERE')

In [ ]:
# ════════════════════════════════════════════════════
# CELL 6: Load Model (4-bit quantized via Unsloth)
# First run downloads ~2GB — subsequent runs use cache
# ════════════════════════════════════════════════════
from unsloth import FastLanguageModel
import torch

print(f"Loading {MODEL_NAME}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,     # Auto-detect: fp16 for T4, bf16 for A100
    load_in_4bit  = True,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded: {total_params/1e9:.2f}B parameters")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ════════════════════════════════════════════════════
# CELL 7: Apply LoRA Adapters
# Only trains ~1-5% of parameters — huge VRAM savings
# ════════════════════════════════════════════════════
model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_R,
    target_modules      = ["q_proj","k_proj","v_proj","o_proj",
                           "gate_proj","up_proj","down_proj"],
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # Saves ~30% VRAM
    random_state        = SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA applied!")
print(f"   Trainable: {trainable:,} params ({100*trainable/total:.2f}% of total)")
print(f"   Frozen:    {total-trainable:,} params")

In [ ]:
# ════════════════════════════════════════════════════
# CELL 8: Load & Format Dataset
# Supports: ShareGPT, Alpaca, OpenAI Chat formats
# ════════════════════════════════════════════════════
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

# Apply model-specific chat template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")
# Use "phi-3" for Phi-3, "qwen-2.5" for Qwen 2.5

# ── Load raw dataset ──────────────────────────────
if DATASET_SOURCE == "huggingface":
    raw = load_dataset(DATASET_NAME, split="train")
    if MAX_SAMPLES: raw = raw.select(range(min(MAX_SAMPLES, len(raw))))
    print(f"Loaded {len(raw)} samples from HuggingFace")

elif DATASET_SOURCE == "local":
    raw = load_dataset("json", data_files=LOCAL_DATASET_PATH, split="train")
    if MAX_SAMPLES: raw = raw.select(range(min(MAX_SAMPLES, len(raw))))
    print(f"Loaded {len(raw)} samples from local JSONL")

elif DATASET_SOURCE == "upload":
    from google.colab import files
    print("Upload your JSONL file:")
    up = files.upload()
    fname = list(up.keys())[0]
    raw = load_dataset("json", data_files=fname, split="train")
    if MAX_SAMPLES: raw = raw.select(range(min(MAX_SAMPLES, len(raw))))
    print(f"Loaded {len(raw)} samples from upload")

# ── Format dataset ────────────────────────────────
def apply_template(examples):
    return {"text": [
        tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        for conv in examples["conversations"]
    ]}

def alpaca_to_chat(examples):
    texts = []
    for inst, inp, out in zip(
        examples["instruction"],
        examples.get("input", [""]*len(examples["instruction"])),
        examples["output"]
    ):
        msgs = [{"role":"system","content":"You are a helpful AI assistant."}]
        user = inst if not inp else f"{inst}\n\n{inp}"
        msgs.append({"role":"user","content":user})
        msgs.append({"role":"assistant","content":out})
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

def openai_to_chat(examples):
    return {"text": [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in examples["messages"]
    ]}

if DATASET_FORMAT == "sharegpt":
    dataset = standardize_sharegpt(raw)
    dataset = dataset.map(apply_template, batched=True, remove_columns=dataset.column_names)
elif DATASET_FORMAT == "alpaca":
    dataset = raw.map(alpaca_to_chat, batched=True, remove_columns=raw.column_names)
elif DATASET_FORMAT == "openai":
    dataset = raw.map(openai_to_chat, batched=True, remove_columns=raw.column_names)

print(f"\n✅ Dataset ready: {len(dataset)} samples")
print(f"\nSample (first 400 chars):")
print(dataset[0]['text'][:400], "...")

In [ ]:
# ════════════════════════════════════════════════════
# CELL 9: 🚀 Train the Model
# ════════════════════════════════════════════════════
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
import os, time

os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    data_collator      = DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8),
    dataset_num_proc   = 2,
    packing            = False,
    args = SFTConfig(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio          = WARMUP_RATIO,
        num_train_epochs      = NUM_EPOCHS,
        max_steps             = MAX_STEPS,
        learning_rate         = LEARNING_RATE,
        fp16                  = not is_bfloat16_supported(),
        bf16                  = is_bfloat16_supported(),
        logging_steps         = 10,
        optim                 = "adamw_8bit",
        weight_decay          = WEIGHT_DECAY,
        lr_scheduler_type     = "cosine",
        seed                  = SEED,
        output_dir            = OUTPUT_DIR,
        save_strategy         = "steps",
        save_steps            = 50,
        report_to             = "none",
        dataset_text_field    = "text",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

sample_batch = trainer.train_dataset[0]
import torch as _torch
labels = _torch.tensor(sample_batch.get("labels", []))
if (labels != -100).sum() == 0:
    raise ValueError(
        "All labels are masked (-100). train_on_responses_only found no assistant tokens.\n"
        "Check that your chat template matches the instruction_part/response_part strings exactly."
    )
print(f"✅ Label sanity check passed: {(labels != -100).sum().item()} trainable tokens in first sample")

print("Starting training...")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Steps: {MAX_STEPS} | LR: {LEARNING_RATE}")
print("─" * 50)

t0 = time.time()
stats = trainer.train()
elapsed = time.time() - t0

print(f"\n✅ Training complete!")
print(f"   Time: {elapsed/60:.1f} minutes")
print(f"   Loss: {stats.training_loss:.4f}")
print(f"   Steps: {stats.global_step}")

In [ ]:
# ════════════════════════════════════════════════════
# CELL 10: Memory & Training Stats
# ════════════════════════════════════════════════════
import torch

gpu = torch.cuda.get_device_properties(0)
peak = round(torch.cuda.max_memory_reserved() / 1e9, 2)
total_vram = round(gpu.total_memory / 1e9, 2)

print(f"Peak VRAM usage: {peak} / {total_vram} GB ({100*peak/total_vram:.1f}%)")

# BUG FIX 5: stats and elapsed only exist if Cell 9 ran successfully.
# Guard against NameError if cells are run out of order.
try:
    print(f"Training speed:  {stats.global_step / (elapsed/60):.1f} steps/min")
    print(f"Final loss:      {stats.training_loss:.4f}")
except NameError:
    print("Run Cell 9 first to see training speed stats.")


In [ ]:
# ════════════════════════════════════════════════════
# CELL 11: Save Model → Drive
# ════════════════════════════════════════════════════
import shutil

# Save LoRA adapters (small: ~50-200MB)
LOCAL_ADAPTER = f"{OUTPUT_DIR}/lora-adapters"
model.save_pretrained(LOCAL_ADAPTER)
tokenizer.save_pretrained(LOCAL_ADAPTER)
print(f"✅ LoRA adapters saved: {LOCAL_ADAPTER}")

# Copy to Google Drive
DRIVE_ADAPTER = f"{DRIVE_SAVE_PATH}/lora-adapters"
shutil.copytree(LOCAL_ADAPTER, DRIVE_ADAPTER, dirs_exist_ok=True)
print(f"✅ Copied to Drive: {DRIVE_ADAPTER}")

# Optional: save merged model (full weights, ~6-7GB)
SAVE_MERGED = False
if SAVE_MERGED:
    merged = f"{OUTPUT_DIR}/merged"
    model.save_pretrained_merged(merged, tokenizer, save_method="merged_16bit")
    shutil.copytree(merged, f"{DRIVE_SAVE_PATH}/merged", dirs_exist_ok=True)
    print(f"✅ Merged model saved to Drive")

In [ ]:
# ════════════════════════════════════════════════════
# CELL 12: Push to HuggingFace Hub
# Set PUSH_TO_HUB = True in Cell 4 to enable
# ════════════════════════════════════════════════════
if PUSH_TO_HUB:
    repo = f"{HF_USERNAME}/{HF_REPO_NAME}"
    print(f"Pushing to: https://huggingface.co/{repo}")
    model.push_to_hub(repo, token=True)
    tokenizer.push_to_hub(repo, token=True)
    print(f"✅ Pushed! https://huggingface.co/{repo}")
    # For merged model:
    # model.push_to_hub_merged(f"{repo}-merged", tokenizer, save_method="merged_16bit", token=True)
else:
    print("ℹ️  Set PUSH_TO_HUB = True in Cell 4 to push to HuggingFace Hub")

In [ ]:
# ════════════════════════════════════════════════════
# CELL 13: 🧪 Inference — Test your fine-tuned model!
# ════════════════════════════════════════════════════
from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_inference(model)  # 2x faster inference

def chat(prompt, system="You are a helpful AI assistant.", max_tokens=512, temp=0.7):
    msgs = [{"role":"system","content":system},{"role":"user","content":prompt}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            input_ids=ids, max_new_tokens=max_tokens,
            temperature=temp, do_sample=(temp > 0),
            top_p=0.9, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)

# Run test prompts
tests = [
    "Tell me about yourself.",
    "What is machine learning in simple terms?",
    "Write a Python function to reverse a string.",
]

for i, prompt in enumerate(tests, 1):
    print(f"\n── Test {i} " + "─"*40)
    print(f"👤 User: {prompt}")
    print(f"🤖 Assistant: {chat(prompt)}")

print("\n✅ Inference tests complete!")

In [ ]:
# ════════════════════════════════════════════════════
# CELL 14: Streaming Inference (ChatGPT-style)
# ════════════════════════════════════════════════════
from transformers import TextStreamer

def chat_stream(prompt, system="You are a helpful AI assistant.", max_tokens=512):
    msgs = [{"role":"system","content":system},{"role":"user","content":prompt}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    print(f"👤 {prompt}\n🤖 ", end="", flush=True)
    with torch.no_grad():
        model.generate(
            input_ids=ids, max_new_tokens=max_tokens,
            temperature=0.7, do_sample=True, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id, streamer=streamer,
        )

chat_stream("Explain LoRA fine-tuning in 3 bullet points.")

In [ ]:
# ════════════════════════════════════════════════════
# CELL 15: Export to GGUF (for Ollama / llama.cpp)
# Set SAVE_GGUF = True to enable
# ════════════════════════════════════════════════════
SAVE_GGUF  = False
GGUF_QUANT = "q4_k_m"  # "q4_k_m" | "q8_0" | "f16"

if SAVE_GGUF:
    print(f"Converting to GGUF [{GGUF_QUANT}]...")
    model.save_pretrained_gguf(f"{OUTPUT_DIR}/gguf", tokenizer, quantization_method=GGUF_QUANT)
    shutil.copytree(f"{OUTPUT_DIR}/gguf", f"{DRIVE_SAVE_PATH}/gguf", dirs_exist_ok=True)
    print(f"✅ GGUF saved to Drive")
    print("   Ollama: ollama create mymodel -f Modelfile")
else:
    print("ℹ️  Set SAVE_GGUF = True to export for Ollama/llama.cpp")

print("\n🎉 All done!")
print(f"   LoRA adapters: {DRIVE_SAVE_PATH}/lora-adapters")
print("   Next: Deploy to HuggingFace Inference Endpoints or vLLM")
print("   Then: Update AI_BASE_URL in your .env.local to point at your model")